# **Practical Q1: Titanic Binary Classification (SVM-RBF)**

### Import Libraries

In [1]:
# Improting needed libraaries

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



---


### Part a) Load Titanic and minimal EDA

In [2]:
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)

# convert target to numeric 0/1 if needed
if y.dtype.name in ("object", "category"):
    y = y.astype("category").cat.codes

print("Shape of X:", X.shape, " | y:", y.shape)
print("Target counts:\n", pd.Series(y).value_counts())

# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# Identify numeric vs categorical columns

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

# Column-wise preprocessing

numeric_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_proc = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_proc, num_cols),
    ("cat", categorical_proc, cat_cols)
], remainder="drop")


Shape of X: (1309, 13)  | y: (1309,)
Target counts:
 0    809
1    500
Name: count, dtype: int64
Numeric columns: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'body']
Categorical columns: ['name', 'sex', 'ticket', 'cabin', 'embarked', 'boat', 'home.dest']




---

### Part (b) SVM (RBF) with hyperparameter tuning
#### (small grid, 3-fold CV, f1_weighted)

In [3]:
pipe = Pipeline([
    ("prep", preprocess),
    ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
])

param_grid = {
    "clf__C": [0.1, 1, 10],
    "clf__gamma": ["scale", 0.1]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=0
)

gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best mean CV (f1_weighted):", gs.best_score_)

best_model = gs.best_estimator_


Best params: {'clf__C': 10, 'clf__gamma': 'scale'}
Best mean CV (f1_weighted): 0.9567775644596616




---

### Part (c) Final evaluation on Test set
#### *numeric only

In [4]:
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("\n=== Test Metrics (SVM-RBF) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-weighted:", f1_score(y_test, y_pred, average="weighted"))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))


=== Test Metrics (SVM-RBF) ===
Accuracy: 0.9541984732824428
F1-weighted: 0.9541076479211636
ROC-AUC: 0.9829938271604939
Confusion matrix:
 [[157   5]
 [  7  93]]

Classification report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.96       162
           1       0.95      0.93      0.94       100

    accuracy                           0.95       262
   macro avg       0.95      0.95      0.95       262
weighted avg       0.95      0.95      0.95       262

